# Task L — AUG Full-Sample In-Sample Optimization

Task L constructs a full-sample in-sample benchmark for
**AUG / SHFE Gold Futures**.

The entire available AUG history is treated as one in-sample dataset, and the
same finalized Channel WithDDControl strategy used in Task K is optimized over
the complete prescribed parameter grid.

The objective is to identify the hindsight-optimal fixed parameter pair under:

**Net Profit / |Maximum Drawdown|**

Task L is not an out-of-sample test. Its role is to provide a full-history
in-sample benchmark against which the rolling OOS results from Task K can later
be compared.

All AUG profit and loss is reported in CNY.

## 0. Imports and Configuration

Task L uses the same finalized AUG data source, contract assumptions, strategy
logic, transaction-cost treatment, and parameter grid as Task K.

The only major design difference is that the entire available AUG history is
treated as a single in-sample optimization period.

In [2]:
import time
import warnings

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from numba import (
    njit,
    prange,
    get_num_threads
)


warnings.filterwarnings(
    "ignore"
)


# ============================================================
# DATA
# ============================================================

CLEAN = (
    Path("data")
    /
    "clean"
)

AUG_PATH = (
    CLEAN
    /
    "AUG.parquet"
)


# ============================================================
# AUG / SHFE GOLD CONTRACT SETTINGS
# ============================================================

PV = 1000.0

SLPG = 0.065

INITIAL_EQUITY = (
    100_000.0
)

CURRENCY = "CNY"


# ============================================================
# EXACT PARAMETER GRID
# ============================================================

CHNLEN_RANGE = np.arange(
    500,
    10001,
    10
)


STPPCT_RANGE = np.round(
    np.arange(
        0.005,
        0.101,
        0.001
    ),
    3
)


N_COMBINATIONS = (
    len(CHNLEN_RANGE)
    *
    len(STPPCT_RANGE)
)


# ============================================================
# CONFIGURATION SUMMARY
# ============================================================

config_table = pd.DataFrame(
    {
        "Setting": [
            "Market",
            "Analysis Type",
            "Currency",
            "Point Value",
            "Round-Turn Slippage",
            "Initial Equity",
            "ChnLen Grid",
            "StpPct Grid",
            "Full Grid Size"
        ],

        "Value": [
            "AUG / SHFE Gold Futures",
            "Full-Sample In-Sample Optimization",
            CURRENCY,
            PV,
            SLPG,
            INITIAL_EQUITY,
            (
                f"500 to 10000 "
                f"(step 10; {len(CHNLEN_RANGE)} values)"
            ),
            (
                f"0.005 to 0.100 "
                f"(step 0.001; {len(STPPCT_RANGE)} values)"
            ),
            f"{N_COMBINATIONS:,}"
        ]
    }
)


print(
    "TASK L — AUG FULL-SAMPLE CONFIGURATION"
)

print(
    "=" * 76
)

display(
    config_table
)


print(
    f"\nNumba threads available: "
    f"{get_num_threads()}"
)

TASK L — AUG FULL-SAMPLE CONFIGURATION


,Setting,Value
0,Market,AUG / SHFE Gold Futures
1,Analysis Type,Full-Sample In-Sample Optimization
2,Currency,CNY
3,Point Value,1000.0
4,Round-Turn Slippage,0.065
5,Initial Equity,100000.0
6,ChnLen Grid,500 to 10000 (step 10; 951 values)
7,StpPct Grid,0.005 to 0.100 (step 0.001; 96 values)
8,Full Grid Size,"91,296"



Numba threads available: 8


## 1. Load and Validate Finalized AUG Data

Task L uses the same finalized clean AUG dataset as Task K.

The full-sample benchmark must be based on exactly the same validated price
history used in the rolling OOS analysis so that any later comparison reflects
only the difference between full-sample in-sample optimization and walk-forward
evaluation, rather than differences in data processing.

Before optimization, the dataset is checked for chronological ordering,
duplicate timestamps, missing OHLC values, and basic OHLC consistency.

In [3]:
aug = pd.read_parquet(
    AUG_PATH
).copy()


# ------------------------------------------------------------
# Standardize timestamp representation
# ------------------------------------------------------------

if "ts" in aug.columns:

    aug["ts"] = pd.to_datetime(
        aug["ts"]
    )

    aug = (
        aug
        .sort_values(
            "ts"
        )
        .reset_index(
            drop=True
        )
    )

    ts_pd = pd.DatetimeIndex(
        aug["ts"]
    )


else:

    aug.index = pd.to_datetime(
        aug.index
    )

    aug = (
        aug
        .sort_index()
    )

    ts_pd = pd.DatetimeIndex(
        aug.index
    )


# ------------------------------------------------------------
# Extract strategy arrays
# ------------------------------------------------------------

Open = aug[
    "Open"
].to_numpy(
    dtype=np.float64
)


High = aug[
    "High"
].to_numpy(
    dtype=np.float64
)


Low = aug[
    "Low"
].to_numpy(
    dtype=np.float64
)


Close = aug[
    "Close"
].to_numpy(
    dtype=np.float64
)


N = len(
    aug
)


# ------------------------------------------------------------
# Data-quality checks
# ------------------------------------------------------------

duplicate_timestamps = int(
    ts_pd
    .duplicated()
    .sum()
)


missing_ohlc = int(
    aug[
        [
            "Open",
            "High",
            "Low",
            "Close"
        ]
    ]
    .isna()
    .sum()
    .sum()
)


chronological = bool(
    ts_pd
    .is_monotonic_increasing
)


ohlc_consistent = bool(
    (
        High
        >=
        np.maximum(
            Open,
            Close
        )
    ).all()

    and

    (
        Low
        <=
        np.minimum(
            Open,
            Close
        )
    ).all()

    and

    (
        High
        >=
        Low
    ).all()
)


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

data_summary = pd.DataFrame(
    {
        "Metric": [
            "Rows",
            "Start",
            "End",
            "Duplicate timestamps",
            "Missing OHLC values",
            "Chronological",
            "OHLC consistent"
        ],

        "Value": [
            f"{N:,}",
            str(
                ts_pd[0]
            ),
            str(
                ts_pd[-1]
            ),
            duplicate_timestamps,
            missing_ohlc,
            chronological,
            ohlc_consistent
        ]
    }
)


print(
    "FINALIZED AUG DATA"
)

print(
    "=" * 76
)

display(
    data_summary
)


# ------------------------------------------------------------
# Hard validation
# ------------------------------------------------------------

assert N > 0

assert chronological

assert duplicate_timestamps == 0

assert missing_ohlc == 0

assert ohlc_consistent


print(
    "\nAll AUG data validation checks passed."
)

FINALIZED AUG DATA


,Metric,Value
0,Rows,"138,744"
1,Start,2018-05-03 09:05:00
2,End,2026-04-10 15:00:00
3,Duplicate timestamps,0
4,Missing OHLC values,0
5,Chronological,True
6,OHLC consistent,True



All AUG data validation checks passed.


## 2. Exact Strategy and Optimization Engine

Task L uses the same finalized Channel WithDDControl implementation as Task K.

The strategy logic, transaction-cost treatment, prior-bar channel construction,
and optimization objective are unchanged. The only difference is that Task L
optimizes the complete AUG history as one in-sample dataset rather than repeating
the optimization across rolling windows.

A monotonic-deque implementation is used to construct prior-bar breakout
channels efficiently, while the stop-percentage dimension is evaluated in
parallel.

In [4]:
# ------------------------------------------------------------
# 2.1 O(N) prior-bar rolling channels
# ------------------------------------------------------------

@njit(cache=True)
def build_channels_deque(
    high,
    low,
    length
):

    n = len(high)


    hh = np.empty(
        n,
        dtype=np.float64
    )

    ll = np.empty(
        n,
        dtype=np.float64
    )


    hh[:] = np.inf

    ll[:] = -np.inf


    max_deque = np.empty(
        n,
        dtype=np.int64
    )

    min_deque = np.empty(
        n,
        dtype=np.int64
    )


    max_head = 0
    max_tail = 0

    min_head = 0
    min_tail = 0


    for k in range(
        n
    ):

        # ----------------------------------------------------
        # Remove observations outside [k-length, k-1]
        # ----------------------------------------------------

        lower_bound = (
            k
            -
            length
        )


        while (
            max_head
            <
            max_tail
            and
            max_deque[
                max_head
            ]
            <
            lower_bound
        ):

            max_head += 1


        while (
            min_head
            <
            min_tail
            and
            min_deque[
                min_head
            ]
            <
            lower_bound
        ):

            min_head += 1


        # ----------------------------------------------------
        # Channel uses PRIOR bars only
        # ----------------------------------------------------

        if k >= length:

            hh[k] = high[
                max_deque[
                    max_head
                ]
            ]

            ll[k] = low[
                min_deque[
                    min_head
                ]
            ]


        # ----------------------------------------------------
        # Add current bar AFTER channel calculation
        # ----------------------------------------------------

        while (
            max_head
            <
            max_tail
            and
            high[
                max_deque[
                    max_tail - 1
                ]
            ]
            <=
            high[k]
        ):

            max_tail -= 1


        max_deque[
            max_tail
        ] = k

        max_tail += 1


        while (
            min_head
            <
            min_tail
            and
            low[
                min_deque[
                    min_tail - 1
                ]
            ]
            >=
            low[k]
        ):

            min_tail -= 1


        min_deque[
            min_tail
        ] = k

        min_tail += 1


    return (
        hh,
        ll
    )


# ------------------------------------------------------------
# 2.2 Exact strategy scorer
# ------------------------------------------------------------

@njit(cache=True)
def score_strategy_fast(
    high,
    low,
    close,
    hh,
    ll,
    length,
    stop_pct,
    slpg,
    pv,
    initial_equity
):

    n = len(
        close
    )


    equity = (
        initial_equity
    )

    equity_max = (
        initial_equity
    )

    min_dd = 0.0


    position = 0

    benchmark_long = 0.0

    benchmark_short = 0.0


    for k in range(
        length,
        n
    ):

        traded = False


        delta = (
            pv
            *
            (
                close[k]
                -
                close[k - 1]
            )
            *
            position
        )


        # ====================================================
        # FLAT
        # ====================================================

        if position == 0:

            buy = (
                high[k]
                >=
                hh[k]
            )

            sell = (
                low[k]
                <=
                ll[k]
            )


            if (
                buy
                and
                sell
            ):

                delta = (
                    -slpg
                    +
                    pv
                    *
                    (
                        ll[k]
                        -
                        hh[k]
                    )
                )


            elif buy:

                delta = (
                    -slpg / 2.0
                    +
                    pv
                    *
                    (
                        close[k]
                        -
                        hh[k]
                    )
                )

                position = 1

                traded = True

                benchmark_long = (
                    high[k]
                )


            elif sell:

                delta = (
                    -slpg / 2.0
                    -
                    pv
                    *
                    (
                        close[k]
                        -
                        ll[k]
                    )
                )

                position = -1

                traded = True

                benchmark_short = (
                    low[k]
                )


        # ====================================================
        # LONG
        # ====================================================

        if (
            position == 1
            and
            not traded
        ):

            sell_short = (
                low[k]
                <=
                ll[k]
            )


            sell = (
                low[k]
                <=
                benchmark_long
                *
                (
                    1.0
                    -
                    stop_pct
                )
            )


            if (
                sell_short
                and
                sell
            ):

                delta = (
                    delta
                    -
                    slpg
                    -
                    2.0
                    *
                    pv
                    *
                    (
                        close[k]
                        -
                        ll[k]
                    )
                )

                position = -1

                benchmark_short = (
                    low[k]
                )


            else:

                if sell:

                    delta = (
                        delta
                        -
                        slpg / 2.0
                        -
                        pv
                        *
                        (
                            close[k]
                            -
                            benchmark_long
                            *
                            (
                                1.0
                                -
                                stop_pct
                            )
                        )
                    )

                    position = 0


                if sell_short:

                    delta = (
                        delta
                        -
                        slpg
                        -
                        2.0
                        *
                        pv
                        *
                        (
                            close[k]
                            -
                            ll[k]
                        )
                    )

                    position = -1

                    benchmark_short = (
                        low[k]
                    )


            if (
                high[k]
                >
                benchmark_long
            ):

                benchmark_long = (
                    high[k]
                )


        # ====================================================
        # SHORT
        # ====================================================

        if (
            position == -1
            and
            not traded
        ):

            buy_long = (
                high[k]
                >=
                hh[k]
            )


            buy = (
                high[k]
                >=
                benchmark_short
                *
                (
                    1.0
                    +
                    stop_pct
                )
            )


            if (
                buy_long
                and
                buy
            ):

                delta = (
                    delta
                    -
                    slpg
                    +
                    2.0
                    *
                    pv
                    *
                    (
                        close[k]
                        -
                        hh[k]
                    )
                )

                position = 1

                benchmark_long = (
                    high[k]
                )


            else:

                if buy:

                    delta = (
                        delta
                        -
                        slpg / 2.0
                        +
                        pv
                        *
                        (
                            close[k]
                            -
                            benchmark_short
                            *
                            (
                                1.0
                                +
                                stop_pct
                            )
                        )
                    )

                    position = 0


                if buy_long:

                    delta = (
                        delta
                        -
                        slpg
                        +
                        2.0
                        *
                        pv
                        *
                        (
                            close[k]
                            -
                            hh[k]
                        )
                    )

                    position = 1

                    benchmark_long = (
                        high[k]
                    )


            if (
                low[k]
                <
                benchmark_short
            ):

                benchmark_short = (
                    low[k]
                )


        # ====================================================
        # EQUITY / DRAWDOWN
        # ====================================================

        equity += (
            delta
        )


        if (
            equity
            >
            equity_max
        ):

            equity_max = (
                equity
            )


        dd = (
            equity
            -
            equity_max
        )


        if (
            dd
            <
            min_dd
        ):

            min_dd = (
                dd
            )


    net_profit = (
        equity
        -
        initial_equity
    )


    if (
        min_dd
        <
        0.0
    ):

        objective = (
            net_profit
            /
            abs(
                min_dd
            )
        )


    elif (
        net_profit
        >
        0.0
    ):

        objective = (
            net_profit
        )


    else:

        objective = (
            -np.inf
        )


    return (
        net_profit,
        min_dd,
        objective
    )


# ------------------------------------------------------------
# 2.3 Parallel StpPct evaluation
# ------------------------------------------------------------

@njit(
    parallel=True,
    cache=True
)
def score_stop_grid_parallel(
    high,
    low,
    close,
    hh,
    ll,
    length,
    stop_grid,
    slpg,
    pv,
    initial_equity
):

    m = len(
        stop_grid
    )


    profits = np.empty(
        m,
        dtype=np.float64
    )

    drawdowns = np.empty(
        m,
        dtype=np.float64
    )

    objectives = np.empty(
        m,
        dtype=np.float64
    )


    for j in prange(
        m
    ):

        (
            profit,
            drawdown,
            objective
        ) = score_strategy_fast(
            high,
            low,
            close,
            hh,
            ll,
            length,
            stop_grid[j],
            slpg,
            pv,
            initial_equity
        )


        profits[j] = (
            profit
        )

        drawdowns[j] = (
            drawdown
        )

        objectives[j] = (
            objective
        )


    return (
        profits,
        drawdowns,
        objectives
    )


print(
    "Exact AUG full-sample strategy engine defined successfully."
)

print(
    f"Channel lengths : "
    f"{len(CHNLEN_RANGE):,}"
)

print(
    f"Stop percentages: "
    f"{len(STPPCT_RANGE):,}"
)

print(
    f"Full grid       : "
    f"{N_COMBINATIONS:,} combinations"
)

Exact AUG full-sample strategy engine defined successfully.
Channel lengths : 951
Stop percentages: 96
Full grid       : 91,296 combinations


## 3. Validate Prior-Bar Channel Construction

Before running the full-sample optimization, the optimized deque-based channel
construction is reconciled against a direct pandas rolling implementation.

This validation confirms that the computationally efficient implementation
produces exactly the same prior-bar breakout channels and does not introduce
look-ahead bias by including the current bar.

In [5]:
# Use a representative channel length from the search grid.
TEST_CHNLEN = 2000


# ------------------------------------------------------------
# Deque implementation
# ------------------------------------------------------------

hh_deque, ll_deque = build_channels_deque(
    High,
    Low,
    TEST_CHNLEN
)


# ------------------------------------------------------------
# Direct pandas reference implementation
#
# shift(1) ensures that the current bar is excluded.
# ------------------------------------------------------------

hh_pandas = (
    pd.Series(
        High
    )
    .shift(1)
    .rolling(
        window=TEST_CHNLEN,
        min_periods=TEST_CHNLEN
    )
    .max()
    .to_numpy(
        dtype=np.float64
    )
)


ll_pandas = (
    pd.Series(
        Low
    )
    .shift(1)
    .rolling(
        window=TEST_CHNLEN,
        min_periods=TEST_CHNLEN
    )
    .min()
    .to_numpy(
        dtype=np.float64
    )
)


# ------------------------------------------------------------
# Compare only the valid channel region
# ------------------------------------------------------------

valid = np.arange(
    N
) >= TEST_CHNLEN


hh_error = float(
    np.max(
        np.abs(
            hh_deque[valid]
            -
            hh_pandas[valid]
        )
    )
)


ll_error = float(
    np.max(
        np.abs(
            ll_deque[valid]
            -
            ll_pandas[valid]
        )
    )
)


hh_equal = bool(
    np.allclose(
        hh_deque[valid],
        hh_pandas[valid],
        rtol=0.0,
        atol=1e-12
    )
)


ll_equal = bool(
    np.allclose(
        ll_deque[valid],
        ll_pandas[valid],
        rtol=0.0,
        atol=1e-12
    )
)


# ------------------------------------------------------------
# Explicit first-valid-bar look-ahead check
# ------------------------------------------------------------

first_valid = (
    TEST_CHNLEN
)


expected_first_hh = float(
    np.max(
        High[
            0:TEST_CHNLEN
        ]
    )
)


expected_first_ll = float(
    np.min(
        Low[
            0:TEST_CHNLEN
        ]
    )
)


first_hh_prior_only = bool(
    np.isclose(
        hh_deque[first_valid],
        expected_first_hh,
        rtol=0.0,
        atol=1e-12
    )
)


first_ll_prior_only = bool(
    np.isclose(
        ll_deque[first_valid],
        expected_first_ll,
        rtol=0.0,
        atol=1e-12
    )
)


# ------------------------------------------------------------
# Validation table
# ------------------------------------------------------------

channel_validation = pd.DataFrame(
    {
        "Check": [
            "HH: deque = pandas rolling",
            "LL: deque = pandas rolling",
            "HH maximum absolute error",
            "LL maximum absolute error",
            "First HH uses prior bars only",
            "First LL uses prior bars only"
        ],

        "Result": [
            hh_equal,
            ll_equal,
            hh_error,
            ll_error,
            first_hh_prior_only,
            first_ll_prior_only
        ]
    }
)


print(
    "CHANNEL CONSTRUCTION VALIDATION"
)

print(
    "=" * 76
)

display(
    channel_validation
)


# ------------------------------------------------------------
# Hard validation
# ------------------------------------------------------------

assert hh_equal
assert ll_equal

assert hh_error <= 1e-12
assert ll_error <= 1e-12

assert first_hh_prior_only
assert first_ll_prior_only


print(
    "\nExact prior-bar channel construction validated."
)

CHANNEL CONSTRUCTION VALIDATION


,Check,Result
0,HH: deque = pandas rolling,True
1,LL: deque = pandas rolling,True
2,HH maximum absolute error,0.0
3,LL maximum absolute error,0.0
4,First HH uses prior bars only,True
5,First LL uses prior bars only,True



Exact prior-bar channel construction validated.


## 4. Define Exact Full-Grid Optimizer

The full AUG history is optimized over the complete parameter grid required by
the project.

For each candidate channel length, the exact prior-bar breakout channels are
constructed once and all stop-percentage candidates are evaluated in parallel.
The optimization criterion is Net Profit divided by the absolute Maximum
Drawdown.

No coarse-to-fine search, parameter pre-screening, or reduced grid is used.
All 91,296 parameter combinations are evaluated.

In [6]:
def optimize_full_grid_final(
    high,
    low,
    close,
    chnlen_range,
    stppct_range,
    slpg,
    pv,
    initial_equity,
    show_progress=True
):

    # --------------------------------------------------------
    # Best-result containers
    # --------------------------------------------------------

    best_objective = -np.inf

    best_chnlen = None
    best_stppct = None

    best_net_profit = None
    best_max_drawdown = None


    total_evaluations = (
        len(chnlen_range)
        *
        len(stppct_range)
    )


    # --------------------------------------------------------
    # Progress iterator
    # --------------------------------------------------------

    if show_progress:

        iterator = tqdm(
            chnlen_range,
            desc="Full-grid optimization"
        )

    else:

        iterator = chnlen_range


    # --------------------------------------------------------
    # Evaluate every ChnLen × StpPct combination
    # --------------------------------------------------------

    for length in iterator:

        length = int(
            length
        )


        # Build exact prior-bar channels once per ChnLen
        hh, ll = build_channels_deque(
            high,
            low,
            length
        )


        # Evaluate all StpPct candidates in parallel
        (
            profits,
            drawdowns,
            objectives
        ) = score_stop_grid_parallel(
            high,
            low,
            close,
            hh,
            ll,
            length,
            stppct_range,
            slpg,
            pv,
            initial_equity
        )


        # ----------------------------------------------------
        # Best StpPct for this ChnLen
        # ----------------------------------------------------

        local_index = int(
            np.argmax(
                objectives
            )
        )


        local_objective = float(
            objectives[
                local_index
            ]
        )


        local_profit = float(
            profits[
                local_index
            ]
        )


        local_drawdown = float(
            drawdowns[
                local_index
            ]
        )


        local_stop = float(
            stppct_range[
                local_index
            ]
        )


        # ----------------------------------------------------
        # Global optimum
        #
        # Tie-breaking:
        #   1. higher objective
        #   2. smaller ChnLen
        #   3. smaller StpPct
        # ----------------------------------------------------

        better = False


        if (
            local_objective
            >
            best_objective
        ):

            better = True


        elif np.isclose(
            local_objective,
            best_objective,
            rtol=0.0,
            atol=1e-12
        ):

            if (
                best_chnlen is None
                or
                length
                <
                best_chnlen
            ):

                better = True


            elif (
                length
                ==
                best_chnlen
                and
                local_stop
                <
                best_stppct
            ):

                better = True


        if better:

            best_objective = (
                local_objective
            )

            best_chnlen = (
                length
            )

            best_stppct = (
                local_stop
            )

            best_net_profit = (
                local_profit
            )

            best_max_drawdown = (
                local_drawdown
            )


    # --------------------------------------------------------
    # Return exact optimum
    # --------------------------------------------------------

    return {
        "ChnLen": int(
            best_chnlen
        ),

        "StpPct": float(
            best_stppct
        ),

        "Net Profit": float(
            best_net_profit
        ),

        "Maximum Drawdown": float(
            best_max_drawdown
        ),

        "Objective": float(
            best_objective
        ),

        "Evaluations": int(
            total_evaluations
        )
    }


print(
    "Exact AUG full-grid optimizer defined successfully."
)

print(
    f"Search space: "
    f"{len(CHNLEN_RANGE):,} × "
    f"{len(STPPCT_RANGE):,} = "
    f"{N_COMBINATIONS:,} combinations."
)

Exact AUG full-grid optimizer defined successfully.
Search space: 951 × 96 = 91,296 combinations.


## 5. Full-Sample Exact Optimization

The entire finalized AUG history is now treated as one in-sample dataset.

All 91,296 combinations of ChnLen and StpPct are evaluated using the exact
strategy engine and the same objective used throughout the project:

**Net Profit / |Maximum Drawdown|**

The resulting parameter pair is a hindsight full-sample optimum and serves only
as an in-sample benchmark for later comparison with the Task K rolling OOS
results.

In [8]:
# ------------------------------------------------------------
# Progress bar
# ------------------------------------------------------------

from tqdm.auto import tqdm


# ------------------------------------------------------------
# Re-define exact full-grid optimizer with tqdm available
# ------------------------------------------------------------

def optimize_full_grid_final(
    high,
    low,
    close,
    chnlen_range,
    stppct_range,
    slpg,
    pv,
    initial_equity,
    show_progress=True
):

    # --------------------------------------------------------
    # Best-result containers
    # --------------------------------------------------------

    best_objective = -np.inf

    best_chnlen = None
    best_stppct = None

    best_net_profit = None
    best_max_drawdown = None


    total_evaluations = (
        len(chnlen_range)
        *
        len(stppct_range)
    )


    # --------------------------------------------------------
    # Progress iterator
    # --------------------------------------------------------

    if show_progress:

        iterator = tqdm(
            chnlen_range,
            desc="Full-grid optimization"
        )

    else:

        iterator = chnlen_range


    # --------------------------------------------------------
    # Evaluate every ChnLen × StpPct combination
    # --------------------------------------------------------

    for length in iterator:

        length = int(
            length
        )


        # ----------------------------------------------------
        # Build exact prior-bar channels once per ChnLen
        # ----------------------------------------------------

        hh, ll = build_channels_deque(
            high,
            low,
            length
        )


        # ----------------------------------------------------
        # Evaluate all StpPct candidates in parallel
        # ----------------------------------------------------

        (
            profits,
            drawdowns,
            objectives
        ) = score_stop_grid_parallel(
            high,
            low,
            close,
            hh,
            ll,
            length,
            stppct_range,
            slpg,
            pv,
            initial_equity
        )


        # ----------------------------------------------------
        # Best StpPct for this ChnLen
        # ----------------------------------------------------

        local_index = int(
            np.argmax(
                objectives
            )
        )


        local_objective = float(
            objectives[
                local_index
            ]
        )


        local_profit = float(
            profits[
                local_index
            ]
        )


        local_drawdown = float(
            drawdowns[
                local_index
            ]
        )


        local_stop = float(
            stppct_range[
                local_index
            ]
        )


        # ----------------------------------------------------
        # Global optimum
        #
        # Tie-breaking:
        #   1. higher objective
        #   2. smaller ChnLen
        #   3. smaller StpPct
        # ----------------------------------------------------

        better = False


        if (
            local_objective
            >
            best_objective
        ):

            better = True


        elif np.isclose(
            local_objective,
            best_objective,
            rtol=0.0,
            atol=1e-12
        ):

            if (
                best_chnlen is None
                or
                length
                <
                best_chnlen
            ):

                better = True


            elif (
                length
                ==
                best_chnlen
                and
                local_stop
                <
                best_stppct
            ):

                better = True


        if better:

            best_objective = (
                local_objective
            )

            best_chnlen = (
                length
            )

            best_stppct = (
                local_stop
            )

            best_net_profit = (
                local_profit
            )

            best_max_drawdown = (
                local_drawdown
            )


    # --------------------------------------------------------
    # Return exact optimum
    # --------------------------------------------------------

    return {
        "ChnLen": int(
            best_chnlen
        ),

        "StpPct": float(
            best_stppct
        ),

        "Net Profit": float(
            best_net_profit
        ),

        "Maximum Drawdown": float(
            best_max_drawdown
        ),

        "Objective": float(
            best_objective
        ),

        "Evaluations": int(
            total_evaluations
        )
    }


# ============================================================
# RUN FULL-SAMPLE OPTIMIZATION
# ============================================================

full_sample_start_time = (
    time.perf_counter()
)


full_sample_optimum = (
    optimize_full_grid_final(
        high=High,
        low=Low,
        close=Close,
        chnlen_range=CHNLEN_RANGE,
        stppct_range=STPPCT_RANGE,
        slpg=SLPG,
        pv=PV,
        initial_equity=INITIAL_EQUITY,
        show_progress=True
    )
)


full_sample_runtime = (
    time.perf_counter()
    -
    full_sample_start_time
)


# ============================================================
# OBJECTIVE RECONCILIATION
# ============================================================

objective_recomputed = (
    full_sample_optimum[
        "Net Profit"
    ]
    /
    abs(
        full_sample_optimum[
            "Maximum Drawdown"
        ]
    )
)


objective_error = abs(
    objective_recomputed
    -
    full_sample_optimum[
        "Objective"
    ]
)


# ============================================================
# RESULT TABLE
# ============================================================

full_sample_optimum_table = pd.DataFrame(
    {
        "Metric": [
            "ChnLen",
            "StpPct",
            "Net Profit",
            "Maximum Drawdown",
            "Objective",
            "Evaluations",
            "Runtime (seconds)",
            "Objective Recalculation Error"
        ],

        "Value": [
            full_sample_optimum[
                "ChnLen"
            ],

            full_sample_optimum[
                "StpPct"
            ],

            full_sample_optimum[
                "Net Profit"
            ],

            full_sample_optimum[
                "Maximum Drawdown"
            ],

            full_sample_optimum[
                "Objective"
            ],

            full_sample_optimum[
                "Evaluations"
            ],

            full_sample_runtime,

            objective_error
        ]
    }
)


print(
    "TASK L — AUG FULL-SAMPLE EXACT OPTIMUM"
)

print(
    "=" * 76
)

print(
    f"Sample: "
    f"{ts_pd[0]} "
    f"-> "
    f"{ts_pd[-1]}"
)

print(
    f"Observations: "
    f"{N:,}"
)

print(
    f"Parameter evaluations: "
    f"{N_COMBINATIONS:,}"
)

print(
    f"Runtime: "
    f"{full_sample_runtime:.2f} seconds"
)

print()


display(
    full_sample_optimum_table
)


# ============================================================
# HARD VALIDATION
# ============================================================

assert (
    full_sample_optimum[
        "ChnLen"
    ]
    in
    CHNLEN_RANGE
)


assert np.any(
    np.isclose(
        STPPCT_RANGE,
        full_sample_optimum[
            "StpPct"
        ],
        rtol=0.0,
        atol=1e-12
    )
)


assert (
    full_sample_optimum[
        "Evaluations"
    ]
    ==
    N_COMBINATIONS
)


assert np.isfinite(
    full_sample_optimum[
        "Objective"
    ]
)


assert np.isclose(
    objective_recomputed,
    full_sample_optimum[
        "Objective"
    ],
    rtol=0.0,
    atol=1e-10
)


print(
    "\nFull-sample AUG optimization validated."
)

Full-grid optimization:   0%|          | 0/951 [00:00<?, ?it/s]

TASK L — AUG FULL-SAMPLE EXACT OPTIMUM
Sample: 2018-05-03 09:05:00 -> 2026-04-10 15:00:00
Observations: 138,744
Parameter evaluations: 91,296
Runtime: 12.48 seconds



,Metric,Value
0,ChnLen,7.100000e+02
1,StpPct,5.000000e-03
2,Net Profit,1.415095e+06
3,Maximum Drawdown,-1.652048e+04
4,Objective,8.565703e+01
5,Evaluations,9.129600e+04
6,Runtime (seconds),1.248137e+01
7,Objective Recalculation Error,0.000000e+00



Full-sample AUG optimization validated.


## 6. Full-Sample Backtest and Engine Reconciliation

The hindsight-optimal full-sample parameter pair is now applied to the complete
AUG history using the same exact bar-level strategy path engine as Task K.

The resulting cumulative P&L must reproduce the Net Profit reported by the
full-grid optimizer.

This reconciliation confirms that parameter optimization and subsequent
full-sample performance analysis use identical strategy accounting.

In [9]:
# ------------------------------------------------------------
# 6.1 Exact bar-level strategy path engine
# ------------------------------------------------------------

@njit(cache=True)
def run_strategy_path_fast(
    high,
    low,
    close,
    length,
    stop_pct,
    slpg,
    pv
):

    n = len(
        close
    )


    hh, ll = build_channels_deque(
        high,
        low,
        length
    )


    pnl = np.zeros(
        n,
        dtype=np.float64
    )


    position_path = np.zeros(
        n,
        dtype=np.int8
    )


    position = 0

    benchmark_long = 0.0
    benchmark_short = 0.0


    for k in range(
        length,
        n
    ):

        traded = False


        delta = (
            pv
            *
            (
                close[k]
                -
                close[k - 1]
            )
            *
            position
        )


        # ====================================================
        # FLAT
        # ====================================================

        if position == 0:

            buy = (
                high[k]
                >=
                hh[k]
            )


            sell = (
                low[k]
                <=
                ll[k]
            )


            if (
                buy
                and
                sell
            ):

                delta = (
                    -slpg
                    +
                    pv
                    *
                    (
                        ll[k]
                        -
                        hh[k]
                    )
                )


            elif buy:

                delta = (
                    -slpg / 2.0
                    +
                    pv
                    *
                    (
                        close[k]
                        -
                        hh[k]
                    )
                )

                position = 1

                traded = True

                benchmark_long = (
                    high[k]
                )


            elif sell:

                delta = (
                    -slpg / 2.0
                    -
                    pv
                    *
                    (
                        close[k]
                        -
                        ll[k]
                    )
                )

                position = -1

                traded = True

                benchmark_short = (
                    low[k]
                )


        # ====================================================
        # LONG
        # ====================================================

        if (
            position == 1
            and
            not traded
        ):

            sell_short = (
                low[k]
                <=
                ll[k]
            )


            sell = (
                low[k]
                <=
                benchmark_long
                *
                (
                    1.0
                    -
                    stop_pct
                )
            )


            if (
                sell_short
                and
                sell
            ):

                delta = (
                    delta
                    -
                    slpg
                    -
                    2.0
                    *
                    pv
                    *
                    (
                        close[k]
                        -
                        ll[k]
                    )
                )

                position = -1

                benchmark_short = (
                    low[k]
                )


            else:

                if sell:

                    delta = (
                        delta
                        -
                        slpg / 2.0
                        -
                        pv
                        *
                        (
                            close[k]
                            -
                            benchmark_long
                            *
                            (
                                1.0
                                -
                                stop_pct
                            )
                        )
                    )

                    position = 0


                if sell_short:

                    delta = (
                        delta
                        -
                        slpg
                        -
                        2.0
                        *
                        pv
                        *
                        (
                            close[k]
                            -
                            ll[k]
                        )
                    )

                    position = -1

                    benchmark_short = (
                        low[k]
                    )


            if (
                high[k]
                >
                benchmark_long
            ):

                benchmark_long = (
                    high[k]
                )


        # ====================================================
        # SHORT
        # ====================================================

        if (
            position == -1
            and
            not traded
        ):

            buy_long = (
                high[k]
                >=
                hh[k]
            )


            buy = (
                high[k]
                >=
                benchmark_short
                *
                (
                    1.0
                    +
                    stop_pct
                )
            )


            if (
                buy_long
                and
                buy
            ):

                delta = (
                    delta
                    -
                    slpg
                    +
                    2.0
                    *
                    pv
                    *
                    (
                        close[k]
                        -
                        hh[k]
                    )
                )

                position = 1

                benchmark_long = (
                    high[k]
                )


            else:

                if buy:

                    delta = (
                        delta
                        -
                        slpg / 2.0
                        +
                        pv
                        *
                        (
                            close[k]
                            -
                            benchmark_short
                            *
                            (
                                1.0
                                +
                                stop_pct
                            )
                        )
                    )

                    position = 0


                if buy_long:

                    delta = (
                        delta
                        -
                        slpg
                        +
                        2.0
                        *
                        pv
                        *
                        (
                            close[k]
                            -
                            hh[k]
                        )
                    )

                    position = 1

                    benchmark_long = (
                        high[k]
                    )


            if (
                low[k]
                <
                benchmark_short
            ):

                benchmark_short = (
                    low[k]
                )


        # ====================================================
        # STORE BAR-LEVEL RESULT
        # ====================================================

        pnl[k] = (
            delta
        )


        position_path[k] = (
            position
        )


    return (
        pnl,
        position_path
    )


# ============================================================
# 6.2 RUN FULL-SAMPLE OPTIMAL PATH
# ============================================================

BEST_CHNLEN = int(
    full_sample_optimum[
        "ChnLen"
    ]
)


BEST_STPPCT = float(
    full_sample_optimum[
        "StpPct"
    ]
)


(
    full_sample_pnl,
    full_sample_position
) = run_strategy_path_fast(
    high=High,
    low=Low,
    close=Close,
    length=BEST_CHNLEN,
    stop_pct=BEST_STPPCT,
    slpg=SLPG,
    pv=PV
)


# ------------------------------------------------------------
# Continuous full-sample equity
# ------------------------------------------------------------

full_sample_equity = (
    INITIAL_EQUITY
    +
    np.cumsum(
        full_sample_pnl
    )
)


path_net_profit = float(
    full_sample_pnl.sum()
)


path_ending_equity = float(
    full_sample_equity[-1]
)


optimizer_net_profit = float(
    full_sample_optimum[
        "Net Profit"
    ]
)


net_profit_error = abs(
    path_net_profit
    -
    optimizer_net_profit
)


# ============================================================
# 6.3 INDEPENDENT SCORER RECONCILIATION
# ============================================================

best_hh, best_ll = (
    build_channels_deque(
        High,
        Low,
        BEST_CHNLEN
    )
)


(
    scorer_net_profit,
    scorer_max_drawdown,
    scorer_objective
) = score_strategy_fast(
    High,
    Low,
    Close,
    best_hh,
    best_ll,
    BEST_CHNLEN,
    BEST_STPPCT,
    SLPG,
    PV,
    INITIAL_EQUITY
)


# ============================================================
# 6.4 RECONCILIATION TABLE
# ============================================================

full_sample_reconciliation = pd.DataFrame(
    {
        "Metric": [
            "ChnLen",
            "StpPct",
            "Optimizer Net Profit",
            "Scorer Net Profit",
            "Path Net Profit",
            "Ending Equity",
            "Path vs Optimizer Absolute Error",
            "Scorer vs Optimizer Absolute Error"
        ],

        "Value": [
            BEST_CHNLEN,
            BEST_STPPCT,
            optimizer_net_profit,
            float(
                scorer_net_profit
            ),
            path_net_profit,
            path_ending_equity,
            net_profit_error,
            abs(
                float(
                    scorer_net_profit
                )
                -
                optimizer_net_profit
            )
        ]
    }
)


print(
    "AUG FULL-SAMPLE ENGINE RECONCILIATION"
)

print(
    "=" * 76
)


display(
    full_sample_reconciliation
)


# ============================================================
# 6.5 HARD VALIDATION
# ============================================================

assert np.isclose(
    scorer_net_profit,
    optimizer_net_profit,
    rtol=0.0,
    atol=1e-8
)


assert np.isclose(
    path_net_profit,
    optimizer_net_profit,
    rtol=0.0,
    atol=1e-8
)


assert np.isclose(
    path_ending_equity,
    INITIAL_EQUITY
    +
    path_net_profit,
    rtol=0.0,
    atol=1e-8
)


print(
    "\nFull-sample optimizer, scorer, and path engine reconcile exactly."
)

AUG FULL-SAMPLE ENGINE RECONCILIATION


,Metric,Value
0,ChnLen,7.100000e+02
1,StpPct,5.000000e-03
2,Optimizer Net Profit,1.415095e+06
3,Scorer Net Profit,1.415095e+06
4,Path Net Profit,1.415095e+06
5,Ending Equity,1.515095e+06
6,Path vs Optimizer Absolute Error,1.629815e-09
7,Scorer vs Optimizer Absolute Error,0.000000e+00



Full-sample optimizer, scorer, and path engine reconcile exactly.


## 7. Full-Sample Performance Metrics

Performance statistics are computed for the hindsight-optimal full-sample
strategy using the complete AUG sample.

These results describe the in-sample performance of the parameter pair selected
using the same full history and therefore should be interpreted as a hindsight
benchmark rather than an out-of-sample estimate of strategy performance.

The resulting metrics will later be compared with the rolling walk-forward OOS
results from Task K.

In [11]:
# ============================================================
# 7.1 BUILD FULL-SAMPLE RESULT FRAME
# ============================================================

full_sample_results = pd.DataFrame(
    {
        "Timestamp": pd.DatetimeIndex(
            ts_pd
        ),

        "PnL": full_sample_pnl,

        "Position": full_sample_position
    }
)


# ------------------------------------------------------------
# Continuous equity
# ------------------------------------------------------------

full_sample_results[
    "Equity"
] = (
    INITIAL_EQUITY
    +
    full_sample_results[
        "PnL"
    ].cumsum()
)


# ============================================================
# 7.2 DRAWDOWN
# ============================================================

full_sample_results[
    "Running_Peak"
] = (
    full_sample_results[
        "Equity"
    ].cummax()
)


full_sample_results[
    "Drawdown_CNY"
] = (
    full_sample_results[
        "Equity"
    ]
    -
    full_sample_results[
        "Running_Peak"
    ]
)


full_sample_results[
    "Drawdown_Pct"
] = (
    full_sample_results[
        "Equity"
    ]
    /
    full_sample_results[
        "Running_Peak"
    ]
    -
    1.0
)


maximum_drawdown_cny = float(
    full_sample_results[
        "Drawdown_CNY"
    ].min()
)


maximum_drawdown_pct = float(
    full_sample_results[
        "Drawdown_Pct"
    ].min()
)


# ============================================================
# 7.3 STARTING / ENDING EQUITY
# ============================================================

starting_equity = float(
    INITIAL_EQUITY
)


ending_equity = float(
    full_sample_results[
        "Equity"
    ].iloc[-1]
)


net_profit = (
    ending_equity
    -
    starting_equity
)


total_return = (
    ending_equity
    /
    starting_equity
    -
    1.0
)


# ============================================================
# 7.4 CAGR
#
# Use actual elapsed calendar time rather than 5-minute
# bar-count annualization.
# ============================================================

start_timestamp = pd.Timestamp(
    full_sample_results[
        "Timestamp"
    ].iloc[0]
)


end_timestamp = pd.Timestamp(
    full_sample_results[
        "Timestamp"
    ].iloc[-1]
)


elapsed_seconds = (
    end_timestamp
    -
    start_timestamp
).total_seconds()


elapsed_years = (
    elapsed_seconds
    /
    (
        365.25
        *
        24.0
        *
        60.0
        *
        60.0
    )
)


cagr = (
    (
        ending_equity
        /
        starting_equity
    )
    **
    (
        1.0
        /
        elapsed_years
    )
    -
    1.0
)


# ============================================================
# 7.5 DAILY EQUITY RETURNS
#
# Use end-of-day equity rather than annualizing 5-minute
# bar returns.
# ============================================================

daily_equity = (
    full_sample_results[
        [
            "Timestamp",
            "Equity"
        ]
    ]
    .copy()
)


daily_equity[
    "Date"
] = (
    daily_equity[
        "Timestamp"
    ]
    .dt
    .normalize()
)


daily_equity = (
    daily_equity
    .groupby(
        "Date",
        as_index=True
    )[
        "Equity"
    ]
    .last()
)


# ------------------------------------------------------------
# Add initial capital before first observed trading day
# so that the first day's return is included.
# ------------------------------------------------------------

daily_equity_with_initial = pd.concat(
    [
        pd.Series(
            [INITIAL_EQUITY],
            index=[
                daily_equity.index[0]
                -
                pd.Timedelta(
                    days=1
                )
            ],
            dtype=np.float64
        ),

        daily_equity.astype(
            np.float64
        )
    ]
)


daily_returns = (
    daily_equity_with_initial
    .pct_change()
    .dropna()
)


daily_returns = (
    daily_returns
    .replace(
        [
            np.inf,
            -np.inf
        ],
        np.nan
    )
    .dropna()
)


# ------------------------------------------------------------
# Daily Sharpe
# ------------------------------------------------------------

if (
    len(
        daily_returns
    )
    >
    1
    and
    daily_returns.std(
        ddof=1
    )
    >
    0
):

    daily_sharpe = float(
        np.sqrt(
            252.0
        )
        *
        daily_returns.mean()
        /
        daily_returns.std(
            ddof=1
        )
    )

else:

    daily_sharpe = np.nan


# ============================================================
# 7.6 CALMAR RATIO
# ============================================================

if (
    maximum_drawdown_pct
    <
    0.0
):

    calmar = float(
        cagr
        /
        abs(
            maximum_drawdown_pct
        )
    )

else:

    calmar = np.nan


# ============================================================
# 7.7 PERFORMANCE SUMMARY
# ============================================================

full_sample_performance = pd.DataFrame(
    {
        "Metric": [
            "Sample Start",
            "Sample End",
            "Elapsed Years",
            "Observations",
            "Trading Days",
            "Starting Equity",
            "Ending Equity",
            "Net Profit",
            "Total Return",
            "CAGR",
            "Maximum Drawdown (CNY)",
            "Maximum Drawdown (%)",
            "Daily Sharpe",
            "Calmar"
        ],

        "Value": [
            start_timestamp,
            end_timestamp,
            elapsed_years,
            len(
                full_sample_results
            ),
            len(
                daily_returns
            ),
            starting_equity,
            ending_equity,
            net_profit,
            total_return,
            cagr,
            maximum_drawdown_cny,
            maximum_drawdown_pct,
            daily_sharpe,
            calmar
        ]
    }
)


print(
    "AUG FULL-SAMPLE PERFORMANCE"
)

print(
    "=" * 76
)


display(
    full_sample_performance
)


# ============================================================
# 7.8 HARD VALIDATION
# ============================================================

# Timestamp checks
assert pd.api.types.is_datetime64_any_dtype(
    full_sample_results[
        "Timestamp"
    ]
)


assert (
    full_sample_results[
        "Timestamp"
    ]
    .is_monotonic_increasing
)


assert (
    full_sample_results[
        "Timestamp"
    ]
    .duplicated()
    .sum()
    ==
    0
)


# ------------------------------------------------------------
# Accounting reconciliation
# ------------------------------------------------------------

assert np.isclose(
    net_profit,
    path_net_profit,
    rtol=0.0,
    atol=1e-8
)


assert np.isclose(
    ending_equity,
    INITIAL_EQUITY
    +
    path_net_profit,
    rtol=0.0,
    atol=1e-8
)


# ------------------------------------------------------------
# Risk-metric checks
# ------------------------------------------------------------

assert (
    maximum_drawdown_cny
    <=
    0.0
)


assert (
    maximum_drawdown_pct
    <=
    0.0
)


assert (
    elapsed_years
    >
    0.0
)


assert np.isfinite(
    cagr
)


assert np.isfinite(
    daily_sharpe
)


assert np.isfinite(
    calmar
)


print(
    "\nFull-sample AUG performance metrics validated."
)

AUG FULL-SAMPLE PERFORMANCE


,Metric,Value
0,Sample Start,2018-05-03 09:05:00
1,Sample End,2026-04-10 15:00:00
2,Elapsed Years,7.937704
3,Observations,138744
4,Trading Days,1927
5,Starting Equity,100000.0
6,Ending Equity,1515095.395
7,Net Profit,1415095.395
8,Total Return,14.150954
9,CAGR,0.408358



Full-sample AUG performance metrics validated.


## 8. Full-Sample Trade-Event Engine Reconciliation

A detailed version of the finalized AUG strategy engine is used to record
trade events and execution prices for the hindsight-optimal full-sample path.

The underlying strategy logic and bar-level P&L accounting are unchanged.
The additional event information is used only to reconstruct completed trades.

Before constructing the trade ledger, the detailed engine is reconciled
bar-by-bar against the previously validated full-sample path engine.

In [12]:
# ------------------------------------------------------------
# 8.1 Detailed event engine
# ------------------------------------------------------------

@njit(cache=True)
def run_strategy_path_events(
    high,
    low,
    close,
    hh,
    ll,
    start_idx,
    stop_pct,
    slpg,
    pv,
    initial_equity
):

    n = len(
        close
    )


    equity_path = np.full(
        n,
        initial_equity,
        dtype=np.float64
    )


    bar_pnl = np.zeros(
        n,
        dtype=np.float64
    )


    position_path = np.zeros(
        n,
        dtype=np.int64
    )


    event_code = np.zeros(
        n,
        dtype=np.int64
    )


    execution_price = np.full(
        n,
        np.nan,
        dtype=np.float64
    )


    position = 0

    benchmark_long = np.nan
    benchmark_short = np.nan

    equity = initial_equity


    for k in range(
        start_idx,
        n
    ):

        delta = (
            pv
            *
            (
                close[k]
                -
                close[k - 1]
            )
            *
            position
        )


        traded = False


        # ====================================================
        # FLAT
        # ====================================================

        if position == 0:

            buy_signal = (
                high[k]
                >=
                hh[k]
            )


            sell_signal = (
                low[k]
                <=
                ll[k]
            )


            if (
                buy_signal
                and
                sell_signal
            ):

                delta = (
                    -slpg
                    +
                    pv
                    *
                    (
                        ll[k]
                        -
                        hh[k]
                    )
                )

                event_code[k] = 1


            elif buy_signal:

                delta = (
                    -slpg / 2.0
                    +
                    pv
                    *
                    (
                        close[k]
                        -
                        hh[k]
                    )
                )

                position = 1

                traded = True

                benchmark_long = (
                    high[k]
                )

                benchmark_short = np.nan

                event_code[k] = 2

                execution_price[k] = (
                    hh[k]
                )


            elif sell_signal:

                delta = (
                    -slpg / 2.0
                    -
                    pv
                    *
                    (
                        close[k]
                        -
                        ll[k]
                    )
                )

                position = -1

                traded = True

                benchmark_short = (
                    low[k]
                )

                benchmark_long = np.nan

                event_code[k] = 3

                execution_price[k] = (
                    ll[k]
                )


        # ====================================================
        # LONG
        # ====================================================

        if (
            position == 1
            and
            not traded
        ):

            channel_reverse = (
                low[k]
                <=
                ll[k]
            )


            stop_price = (
                benchmark_long
                *
                (
                    1.0
                    -
                    stop_pct
                )
            )


            stop_trigger = (
                low[k]
                <=
                stop_price
            )


            if (
                channel_reverse
                and
                stop_trigger
            ):

                delta = (
                    delta
                    -
                    slpg
                    -
                    2.0
                    *
                    pv
                    *
                    (
                        close[k]
                        -
                        ll[k]
                    )
                )

                position = -1

                benchmark_short = (
                    low[k]
                )

                event_code[k] = 4

                execution_price[k] = (
                    ll[k]
                )


            else:

                if stop_trigger:

                    delta = (
                        delta
                        -
                        slpg / 2.0
                        -
                        pv
                        *
                        (
                            close[k]
                            -
                            stop_price
                        )
                    )

                    position = 0

                    event_code[k] = 5

                    execution_price[k] = (
                        stop_price
                    )


                if channel_reverse:

                    delta = (
                        delta
                        -
                        slpg
                        -
                        2.0
                        *
                        pv
                        *
                        (
                            close[k]
                            -
                            ll[k]
                        )
                    )

                    position = -1

                    benchmark_short = (
                        low[k]
                    )

                    event_code[k] = 4

                    execution_price[k] = (
                        ll[k]
                    )


            benchmark_long = max(
                high[k],
                benchmark_long
            )


        # ====================================================
        # SHORT
        # ====================================================

        if (
            position == -1
            and
            not traded
        ):

            channel_reverse = (
                high[k]
                >=
                hh[k]
            )


            stop_price = (
                benchmark_short
                *
                (
                    1.0
                    +
                    stop_pct
                )
            )


            stop_trigger = (
                high[k]
                >=
                stop_price
            )


            if (
                channel_reverse
                and
                stop_trigger
            ):

                delta = (
                    delta
                    -
                    slpg
                    +
                    2.0
                    *
                    pv
                    *
                    (
                        close[k]
                        -
                        hh[k]
                    )
                )

                position = 1

                benchmark_long = (
                    high[k]
                )

                event_code[k] = 6

                execution_price[k] = (
                    hh[k]
                )


            else:

                if stop_trigger:

                    delta = (
                        delta
                        -
                        slpg / 2.0
                        +
                        pv
                        *
                        (
                            close[k]
                            -
                            stop_price
                        )
                    )

                    position = 0

                    event_code[k] = 7

                    execution_price[k] = (
                        stop_price
                    )


                if channel_reverse:

                    delta = (
                        delta
                        -
                        slpg
                        +
                        2.0
                        *
                        pv
                        *
                        (
                            close[k]
                            -
                            hh[k]
                        )
                    )

                    position = 1

                    benchmark_long = (
                        high[k]
                    )

                    event_code[k] = 6

                    execution_price[k] = (
                        hh[k]
                    )


            benchmark_short = min(
                low[k],
                benchmark_short
            )


        # ====================================================
        # STORE BAR RESULT
        # ====================================================

        equity += (
            delta
        )


        equity_path[k] = (
            equity
        )


        bar_pnl[k] = (
            delta
        )


        position_path[k] = (
            position
        )


    return (
        equity_path,
        bar_pnl,
        position_path,
        event_code,
        execution_price
    )


# ============================================================
# 8.2 RUN DETAILED FULL-SAMPLE PATH
# ============================================================

(
    full_event_equity,
    full_event_pnl,
    full_event_position,
    full_event_code,
    full_execution_price
) = run_strategy_path_events(
    high=High,
    low=Low,
    close=Close,
    hh=best_hh,
    ll=best_ll,
    start_idx=BEST_CHNLEN,
    stop_pct=BEST_STPPCT,
    slpg=SLPG,
    pv=PV,
    initial_equity=INITIAL_EQUITY
)


# ============================================================
# 8.3 RECONCILIATION
# ============================================================

pnl_max_error = float(
    np.max(
        np.abs(
            full_event_pnl
            -
            full_sample_pnl
        )
    )
)


position_match = bool(
    np.array_equal(
        full_event_position,
        full_sample_position
    )
)


event_total_pnl = float(
    full_event_pnl.sum()
)


event_total_pnl_error = abs(
    event_total_pnl
    -
    path_net_profit
)


event_count = int(
    (
        full_event_code
        !=
        0
    )
    .sum()
)


full_event_reconciliation = pd.DataFrame(
    {
        "Check": [
            "Maximum bar-level P&L error",
            "Total P&L error",
            "Position paths identical",
            "Recorded trade events"
        ],

        "Value": [
            pnl_max_error,
            event_total_pnl_error,
            position_match,
            event_count
        ]
    }
)


print(
    "AUG FULL-SAMPLE TRADE-EVENT ENGINE RECONCILIATION"
)

print(
    "=" * 76
)


display(
    full_event_reconciliation
)


# ============================================================
# 8.4 HARD VALIDATION
# ============================================================

assert np.allclose(
    full_event_pnl,
    full_sample_pnl,
    rtol=0.0,
    atol=1e-8
)


assert (
    position_match
)


assert np.isclose(
    event_total_pnl,
    path_net_profit,
    rtol=0.0,
    atol=1e-8
)


print(
    "\nDetailed full-sample trade-event engine reconciles exactly."
)

AUG FULL-SAMPLE TRADE-EVENT ENGINE RECONCILIATION


,Check,Value
0,Maximum bar-level P&L error,0.0
1,Total P&L error,0.0
2,Position paths identical,True
3,Recorded trade events,831



Detailed full-sample trade-event engine reconciles exactly.


## 9. Full-Sample Completed Trade Ledger

The detailed event path is used to reconstruct completed full-sample trades
under the hindsight-optimal parameter pair.

A completed trade begins when a new position is opened from flat or by reversal
and ends when that position is closed by a trailing stop or opposite-channel
reversal.

Because Task L is a single continuous full-sample path rather than a sequence of
walk-forward windows, there are no quarterly boundary-attribution complications.
Only a trade that remains open at the very end of the sample is excluded from
completed-trade statistics.

In [13]:
# ============================================================
# 9.1 BUILD FULL-SAMPLE COMPLETED TRADE LEDGER
# ============================================================

trade_rows = []

same_bar_flat_event_count = 0


# ------------------------------------------------------------
# Active trade state
# ------------------------------------------------------------

active_trade = False

entry_time = None

entry_price = np.nan

entry_direction = 0

entry_bar = None


# ============================================================
# PROCESS FULL-SAMPLE EVENTS
# ============================================================

for k in range(
    len(
        full_event_code
    )
):

    code = int(
        full_event_code[k]
    )


    if code == 0:

        continue


    px = float(
        full_execution_price[k]
    )


    ts = pd.Timestamp(
        ts_pd[k]
    )


    # ========================================================
    # CODE 1 — Both channels crossed while flat
    # ========================================================

    if code == 1:

        same_bar_flat_event_count += 1


    # ========================================================
    # CODE 2 — Flat -> Long
    # ========================================================

    elif code == 2:

        active_trade = True

        entry_time = ts

        entry_price = px

        entry_direction = 1

        entry_bar = k


    # ========================================================
    # CODE 3 — Flat -> Short
    # ========================================================

    elif code == 3:

        active_trade = True

        entry_time = ts

        entry_price = px

        entry_direction = -1

        entry_bar = k


    # ========================================================
    # CODE 4 — Long -> Short Reversal
    # ========================================================

    elif code == 4:

        # ----------------------------------------------------
        # Close active long trade
        # ----------------------------------------------------

        if (
            active_trade
            and
            entry_direction == 1
        ):

            exit_price = px


            gross_pnl = (
                PV
                *
                (
                    exit_price
                    -
                    entry_price
                )
            )


            transaction_cost = (
                SLPG
            )


            net_pnl_trade = (
                gross_pnl
                -
                transaction_cost
            )


            trade_rows.append(
                {
                    "Direction": "Long",

                    "Entry_Time": (
                        entry_time
                    ),

                    "Exit_Time": (
                        ts
                    ),

                    "Entry_Price": (
                        entry_price
                    ),

                    "Exit_Price": (
                        exit_price
                    ),

                    "Holding_Bars": (
                        k
                        -
                        entry_bar
                    ),

                    "Gross_PnL": (
                        gross_pnl
                    ),

                    "Transaction_Cost": (
                        transaction_cost
                    ),

                    "Net_PnL": (
                        net_pnl_trade
                    ),

                    "Exit_Reason": (
                        "Channel Reversal"
                    )
                }
            )


        # ----------------------------------------------------
        # Same bar opens new short trade
        # ----------------------------------------------------

        active_trade = True

        entry_time = ts

        entry_price = px

        entry_direction = -1

        entry_bar = k


    # ========================================================
    # CODE 5 — Long Trailing-Stop Exit
    # ========================================================

    elif code == 5:

        if (
            active_trade
            and
            entry_direction == 1
        ):

            exit_price = px


            gross_pnl = (
                PV
                *
                (
                    exit_price
                    -
                    entry_price
                )
            )


            transaction_cost = (
                SLPG
            )


            net_pnl_trade = (
                gross_pnl
                -
                transaction_cost
            )


            trade_rows.append(
                {
                    "Direction": "Long",

                    "Entry_Time": (
                        entry_time
                    ),

                    "Exit_Time": (
                        ts
                    ),

                    "Entry_Price": (
                        entry_price
                    ),

                    "Exit_Price": (
                        exit_price
                    ),

                    "Holding_Bars": (
                        k
                        -
                        entry_bar
                    ),

                    "Gross_PnL": (
                        gross_pnl
                    ),

                    "Transaction_Cost": (
                        transaction_cost
                    ),

                    "Net_PnL": (
                        net_pnl_trade
                    ),

                    "Exit_Reason": (
                        "Trailing Stop"
                    )
                }
            )


        active_trade = False

        entry_time = None

        entry_price = np.nan

        entry_direction = 0

        entry_bar = None


    # ========================================================
    # CODE 6 — Short -> Long Reversal
    # ========================================================

    elif code == 6:

        # ----------------------------------------------------
        # Close active short trade
        # ----------------------------------------------------

        if (
            active_trade
            and
            entry_direction == -1
        ):

            exit_price = px


            gross_pnl = (
                PV
                *
                (
                    entry_price
                    -
                    exit_price
                )
            )


            transaction_cost = (
                SLPG
            )


            net_pnl_trade = (
                gross_pnl
                -
                transaction_cost
            )


            trade_rows.append(
                {
                    "Direction": "Short",

                    "Entry_Time": (
                        entry_time
                    ),

                    "Exit_Time": (
                        ts
                    ),

                    "Entry_Price": (
                        entry_price
                    ),

                    "Exit_Price": (
                        exit_price
                    ),

                    "Holding_Bars": (
                        k
                        -
                        entry_bar
                    ),

                    "Gross_PnL": (
                        gross_pnl
                    ),

                    "Transaction_Cost": (
                        transaction_cost
                    ),

                    "Net_PnL": (
                        net_pnl_trade
                    ),

                    "Exit_Reason": (
                        "Channel Reversal"
                    )
                }
            )


        # ----------------------------------------------------
        # Same bar opens new long trade
        # ----------------------------------------------------

        active_trade = True

        entry_time = ts

        entry_price = px

        entry_direction = 1

        entry_bar = k


    # ========================================================
    # CODE 7 — Short Trailing-Stop Exit
    # ========================================================

    elif code == 7:

        if (
            active_trade
            and
            entry_direction == -1
        ):

            exit_price = px


            gross_pnl = (
                PV
                *
                (
                    entry_price
                    -
                    exit_price
                )
            )


            transaction_cost = (
                SLPG
            )


            net_pnl_trade = (
                gross_pnl
                -
                transaction_cost
            )


            trade_rows.append(
                {
                    "Direction": "Short",

                    "Entry_Time": (
                        entry_time
                    ),

                    "Exit_Time": (
                        ts
                    ),

                    "Entry_Price": (
                        entry_price
                    ),

                    "Exit_Price": (
                        exit_price
                    ),

                    "Holding_Bars": (
                        k
                        -
                        entry_bar
                    ),

                    "Gross_PnL": (
                        gross_pnl
                    ),

                    "Transaction_Cost": (
                        transaction_cost
                    ),

                    "Net_PnL": (
                        net_pnl_trade
                    ),

                    "Exit_Reason": (
                        "Trailing Stop"
                    )
                }
            )


        active_trade = False

        entry_time = None

        entry_price = np.nan

        entry_direction = 0

        entry_bar = None


# ============================================================
# ASSEMBLE LEDGER
# ============================================================

full_sample_trade_ledger = pd.DataFrame(
    trade_rows
)


incomplete_trade_at_end = bool(
    active_trade
)


# ------------------------------------------------------------
# Basic validation
# ------------------------------------------------------------

assert len(
    full_sample_trade_ledger
) > 0


assert (
    full_sample_trade_ledger[
        "Exit_Time"
    ]
    >=
    full_sample_trade_ledger[
        "Entry_Time"
    ]
).all()


assert (
    full_sample_trade_ledger[
        "Holding_Bars"
    ]
    >=
    0
).all()


assert np.isfinite(
    full_sample_trade_ledger[
        [
            "Entry_Price",
            "Exit_Price",
            "Gross_PnL",
            "Transaction_Cost",
            "Net_PnL"
        ]
    ]
    .to_numpy()
).all()


# ------------------------------------------------------------
# Accounting summary
# ------------------------------------------------------------

completed_trade_net_pnl = float(
    full_sample_trade_ledger[
        "Net_PnL"
    ].sum()
)


remaining_attribution = (
    path_net_profit
    -
    completed_trade_net_pnl
)


remaining_attribution_share = (
    remaining_attribution
    /
    path_net_profit
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print(
    "AUG FULL-SAMPLE COMPLETED TRADE LEDGER"
)

print(
    "=" * 76
)


print(
    f"Completed trades              : "
    f"{len(full_sample_trade_ledger):,}"
)


print(
    f"Same-bar flat channel events  : "
    f"{same_bar_flat_event_count:,}"
)


print(
    f"Incomplete trade at sample end: "
    f"{incomplete_trade_at_end}"
)


print(
    f"Completed-trade Net P&L       : "
    f"{completed_trade_net_pnl:,.2f} CNY"
)


print(
    f"Remaining attribution         : "
    f"{remaining_attribution:,.2f} CNY"
)


print(
    f"Remaining attribution share   : "
    f"{remaining_attribution_share:.2%}"
)


print(
    "\nFirst 10 completed trades:"
)


display(
    full_sample_trade_ledger.head(
        10
    )
)


print(
    "\nFull-sample completed trade ledger constructed successfully."
)

AUG FULL-SAMPLE COMPLETED TRADE LEDGER
Completed trades              : 417
Same-bar flat channel events  : 0
Incomplete trade at sample end: False
Completed-trade Net P&L       : 1,415,095.40 CNY
Remaining attribution         : 0.00 CNY
Remaining attribution share   : 0.00%

First 10 completed trades:


,Direction,Entry_Time,Exit_Time,Entry_Price,Exit_Price,Holding_Bars,Gross_PnL,Transaction_Cost,Net_PnL,Exit_Reason
0,Short,2018-05-17 09:05:00,2018-05-22 09:05:00,270.35,270.79725,216,-447.25,0.065,-447.315,Trailing Stop
1,Long,2018-05-29 14:15:00,2018-06-01 09:25:00,273.65,273.32650,158,-323.50,0.065,-323.565,Trailing Stop
2,Long,2018-06-15 09:05:00,2018-06-19 09:05:00,274.00,271.65000,72,-2350.00,0.065,-2350.065,Channel Reversal
3,Short,2018-06-19 09:05:00,2018-06-22 10:10:00,271.65,270.44550,229,1204.50,0.065,1204.435,Trailing Stop
4,Long,2018-07-02 14:35:00,2018-07-10 09:05:00,272.05,272.33150,366,281.50,0.065,281.435,Trailing Stop
5,Short,2018-07-17 09:05:00,2018-07-19 10:35:00,271.60,269.94300,162,1657.00,0.065,1656.935,Trailing Stop
6,Short,2018-08-03 09:05:00,2018-08-13 09:05:00,270.45,269.84250,432,607.50,0.065,607.435,Trailing Stop
7,Short,2018-08-14 09:05:00,2018-08-16 10:35:00,268.50,264.01350,162,4486.50,0.065,4486.435,Trailing Stop
8,Long,2018-08-28 13:35:00,2018-08-29 09:05:00,267.90,267.00825,18,-891.75,0.065,-891.815,Trailing Stop
9,Short,2018-09-04 15:00:00,2018-09-06 10:35:00,265.55,266.52600,91,-976.00,0.065,-976.065,Trailing Stop



Full-sample completed trade ledger constructed successfully.


### 9.2 Full-Sample Trade Statistics

Completed trades are summarized using win rate, profit factor, payoff ratio,
expectancy, holding period, direction-level attribution, and exit-reason counts.

Because the full-sample path contains no incomplete terminal trade, completed
trade P&L reconciles exactly to total portfolio P&L.

In [14]:
ledger = (
    full_sample_trade_ledger
    .copy()
)


# ------------------------------------------------------------
# 1. Basic trade counts
# ------------------------------------------------------------

completed_trades = len(
    ledger
)


winning_trades = int(
    (
        ledger[
            "Net_PnL"
        ]
        >
        0.0
    ).sum()
)


losing_trades = int(
    (
        ledger[
            "Net_PnL"
        ]
        <
        0.0
    ).sum()
)


flat_trades = int(
    (
        ledger[
            "Net_PnL"
        ]
        ==
        0.0
    ).sum()
)


win_rate = (
    winning_trades
    /
    completed_trades
)


# ------------------------------------------------------------
# 2. Profit / loss decomposition
# ------------------------------------------------------------

gross_profit = float(
    ledger.loc[
        ledger[
            "Net_PnL"
        ]
        >
        0.0,
        "Net_PnL"
    ].sum()
)


gross_loss = float(
    ledger.loc[
        ledger[
            "Net_PnL"
        ]
        <
        0.0,
        "Net_PnL"
    ].sum()
)


profit_factor = (
    gross_profit
    /
    abs(
        gross_loss
    )
)


average_winner = float(
    ledger.loc[
        ledger[
            "Net_PnL"
        ]
        >
        0.0,
        "Net_PnL"
    ].mean()
)


average_loser = float(
    ledger.loc[
        ledger[
            "Net_PnL"
        ]
        <
        0.0,
        "Net_PnL"
    ].mean()
)


payoff_ratio = (
    average_winner
    /
    abs(
        average_loser
    )
)


expectancy_per_trade = float(
    ledger[
        "Net_PnL"
    ].mean()
)


median_trade_pnl = float(
    ledger[
        "Net_PnL"
    ].median()
)


# ------------------------------------------------------------
# 3. Holding-period statistics
# ------------------------------------------------------------

average_holding_bars = float(
    ledger[
        "Holding_Bars"
    ].mean()
)


median_holding_bars = float(
    ledger[
        "Holding_Bars"
    ].median()
)


maximum_holding_bars = int(
    ledger[
        "Holding_Bars"
    ].max()
)


# ------------------------------------------------------------
# 4. Direction statistics
# ------------------------------------------------------------

long_mask = (
    ledger[
        "Direction"
    ]
    ==
    "Long"
)


short_mask = (
    ledger[
        "Direction"
    ]
    ==
    "Short"
)


long_trades = int(
    long_mask.sum()
)


short_trades = int(
    short_mask.sum()
)


long_net_pnl = float(
    ledger.loc[
        long_mask,
        "Net_PnL"
    ].sum()
)


short_net_pnl = float(
    ledger.loc[
        short_mask,
        "Net_PnL"
    ].sum()
)


# ------------------------------------------------------------
# 5. Exit-reason statistics
# ------------------------------------------------------------

trailing_stop_exits = int(
    (
        ledger[
            "Exit_Reason"
        ]
        ==
        "Trailing Stop"
    ).sum()
)


channel_reversal_exits = int(
    (
        ledger[
            "Exit_Reason"
        ]
        ==
        "Channel Reversal"
    ).sum()
)


# ------------------------------------------------------------
# 6. Assemble statistics table
# ------------------------------------------------------------

full_sample_trade_statistics = pd.DataFrame(
    {
        "Metric": [
            "Completed Trades",
            "Winning Trades",
            "Losing Trades",
            "Flat Trades",
            "Win Rate",
            "Gross Profit",
            "Gross Loss",
            "Profit Factor",
            "Average Winner",
            "Average Loser",
            "Payoff Ratio",
            "Expectancy per Trade",
            "Median Trade P&L",
            "Average Holding Bars",
            "Median Holding Bars",
            "Maximum Holding Bars",
            "Long Trades",
            "Short Trades",
            "Long Net P&L",
            "Short Net P&L",
            "Trailing-Stop Exits",
            "Channel-Reversal Exits"
        ],

        "Value": [
            completed_trades,
            winning_trades,
            losing_trades,
            flat_trades,
            win_rate,
            gross_profit,
            gross_loss,
            profit_factor,
            average_winner,
            average_loser,
            payoff_ratio,
            expectancy_per_trade,
            median_trade_pnl,
            average_holding_bars,
            median_holding_bars,
            maximum_holding_bars,
            long_trades,
            short_trades,
            long_net_pnl,
            short_net_pnl,
            trailing_stop_exits,
            channel_reversal_exits
        ]
    }
)


# ============================================================
# 7. VALIDATION
# ============================================================

trade_pnl_reconciliation_error = abs(
    float(
        ledger[
            "Net_PnL"
        ].sum()
    )
    -
    path_net_profit
)


direction_pnl_error = abs(
    (
        long_net_pnl
        +
        short_net_pnl
    )
    -
    float(
        ledger[
            "Net_PnL"
        ].sum()
    )
)


trade_count_error = abs(
    (
        winning_trades
        +
        losing_trades
        +
        flat_trades
    )
    -
    completed_trades
)


direction_count_error = abs(
    (
        long_trades
        +
        short_trades
    )
    -
    completed_trades
)


exit_count_error = abs(
    (
        trailing_stop_exits
        +
        channel_reversal_exits
    )
    -
    completed_trades
)


# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

assert (
    trade_pnl_reconciliation_error
    <
    1e-6
)


assert (
    direction_pnl_error
    <
    1e-6
)


assert (
    trade_count_error
    ==
    0
)


assert (
    direction_count_error
    ==
    0
)


assert (
    exit_count_error
    ==
    0
)


assert (
    0.0
    <=
    win_rate
    <=
    1.0
)


assert (
    profit_factor
    >
    0.0
)


assert (
    payoff_ratio
    >
    0.0
)


# ============================================================
# 8. DISPLAY
# ============================================================

print(
    "AUG FULL-SAMPLE COMPLETED TRADE STATISTICS"
)

print(
    "=" * 76
)


display(
    full_sample_trade_statistics
)


print(
    "\nTRADE-STATISTICS VALIDATION"
)

print(
    "=" * 76
)


print(
    f"Trade P&L reconciliation error : "
    f"{trade_pnl_reconciliation_error:.12f}"
)


print(
    f"Direction P&L error            : "
    f"{direction_pnl_error:.12f}"
)


print(
    f"Trade-count error              : "
    f"{trade_count_error}"
)


print(
    f"Direction-count error          : "
    f"{direction_count_error}"
)


print(
    f"Exit-count error               : "
    f"{exit_count_error}"
)


print(
    "\nAll full-sample completed-trade statistics validated."
)

AUG FULL-SAMPLE COMPLETED TRADE STATISTICS


,Metric,Value
0,Completed Trades,4.170000e+02
1,Winning Trades,2.530000e+02
2,Losing Trades,1.640000e+02
3,Flat Trades,0.000000e+00
4,Win Rate,6.067146e-01
5,Gross Profit,1.635679e+06
6,Gross Loss,-2.205835e+05
7,Profit Factor,7.415238e+00
8,Average Winner,6.465134e+03
9,Average Loser,-1.345021e+03



TRADE-STATISTICS VALIDATION
Trade P&L reconciliation error : 0.000000000000
Direction P&L error            : 0.000000000000
Trade-count error              : 0
Direction-count error          : 0
Exit-count error               : 0

All full-sample completed-trade statistics validated.


## 10. Full-Sample Completed-Trade Profit Concentration

Profit concentration is examined to determine whether the full-sample
hindsight benchmark is dominated by a small number of unusually profitable
trades.

The analysis reports the contribution of the largest winning trades to gross
winning P&L and completed-trade net P&L, together with counterfactual net P&L
after removing the largest winners.

In [15]:
ledger = (
    full_sample_trade_ledger
    .copy()
)


# ------------------------------------------------------------
# 1. Winning trades
# ------------------------------------------------------------

winning_ledger = (
    ledger.loc[
        ledger[
            "Net_PnL"
        ]
        >
        0.0
    ]
    .sort_values(
        "Net_PnL",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


assert len(
    winning_ledger
) > 0


# ------------------------------------------------------------
# 2. Core totals
# ------------------------------------------------------------

gross_winning_pnl = float(
    winning_ledger[
        "Net_PnL"
    ].sum()
)


completed_net_pnl = float(
    ledger[
        "Net_PnL"
    ].sum()
)


largest_winning_trade = float(
    winning_ledger[
        "Net_PnL"
    ].iloc[0]
)


# ------------------------------------------------------------
# 3. Concentration helper
# ------------------------------------------------------------

def top_n_winner_pnl(
    n
):

    return float(
        winning_ledger[
            "Net_PnL"
        ]
        .head(
            n
        )
        .sum()
    )


top_1_pnl = top_n_winner_pnl(
    1
)

top_3_pnl = top_n_winner_pnl(
    3
)

top_5_pnl = top_n_winner_pnl(
    5
)

top_10_pnl = top_n_winner_pnl(
    10
)


# ------------------------------------------------------------
# 4. Shares of gross winning P&L
# ------------------------------------------------------------

top_1_share_gross_winners = (
    top_1_pnl
    /
    gross_winning_pnl
)


top_3_share_gross_winners = (
    top_3_pnl
    /
    gross_winning_pnl
)


top_5_share_gross_winners = (
    top_5_pnl
    /
    gross_winning_pnl
)


top_10_share_gross_winners = (
    top_10_pnl
    /
    gross_winning_pnl
)


# ------------------------------------------------------------
# 5. Shares of completed net P&L
# ------------------------------------------------------------

top_1_share_completed_net = (
    top_1_pnl
    /
    completed_net_pnl
)


top_3_share_completed_net = (
    top_3_pnl
    /
    completed_net_pnl
)


top_5_share_completed_net = (
    top_5_pnl
    /
    completed_net_pnl
)


top_10_share_completed_net = (
    top_10_pnl
    /
    completed_net_pnl
)


# ------------------------------------------------------------
# 6. Net P&L after removing largest winners
# ------------------------------------------------------------

net_excluding_top_1 = (
    completed_net_pnl
    -
    top_1_pnl
)


net_excluding_top_3 = (
    completed_net_pnl
    -
    top_3_pnl
)


net_excluding_top_5 = (
    completed_net_pnl
    -
    top_5_pnl
)


net_excluding_top_10 = (
    completed_net_pnl
    -
    top_10_pnl
)


# ------------------------------------------------------------
# 7. Assemble concentration table
# ------------------------------------------------------------

full_sample_profit_concentration = pd.DataFrame(
    {
        "Metric": [
            "Largest Winning Trade",
            "Top 1 Share of Gross Winning P&L",
            "Top 3 Share of Gross Winning P&L",
            "Top 5 Share of Gross Winning P&L",
            "Top 10 Share of Gross Winning P&L",
            "Top 1 Share of Completed Net P&L",
            "Top 3 Share of Completed Net P&L",
            "Top 5 Share of Completed Net P&L",
            "Top 10 Share of Completed Net P&L",
            "Net P&L Excluding Top 1 Winner",
            "Net P&L Excluding Top 3 Winners",
            "Net P&L Excluding Top 5 Winners",
            "Net P&L Excluding Top 10 Winners"
        ],

        "Value": [
            largest_winning_trade,
            top_1_share_gross_winners,
            top_3_share_gross_winners,
            top_5_share_gross_winners,
            top_10_share_gross_winners,
            top_1_share_completed_net,
            top_3_share_completed_net,
            top_5_share_completed_net,
            top_10_share_completed_net,
            net_excluding_top_1,
            net_excluding_top_3,
            net_excluding_top_5,
            net_excluding_top_10
        ]
    }
)


# ------------------------------------------------------------
# 8. Validation
# ------------------------------------------------------------

assert (
    largest_winning_trade
    ==
    float(
        winning_ledger[
            "Net_PnL"
        ].max()
    )
)


assert (
    0.0
    <
    top_1_share_gross_winners
    <=
    top_3_share_gross_winners
    <=
    top_5_share_gross_winners
    <=
    top_10_share_gross_winners
    <=
    1.0
)


assert (
    top_1_pnl
    <=
    top_3_pnl
    <=
    top_5_pnl
    <=
    top_10_pnl
)


assert abs(
    completed_net_pnl
    -
    path_net_profit
) < 1e-6


# ------------------------------------------------------------
# 9. Top 10 winning trades
# ------------------------------------------------------------

top_10_winning_trades = (
    winning_ledger[
        [
            "Direction",
            "Entry_Time",
            "Exit_Time",
            "Holding_Bars",
            "Net_PnL",
            "Exit_Reason"
        ]
    ]
    .head(
        10
    )
    .copy()
)


# ============================================================
# 10. DISPLAY
# ============================================================

print(
    "AUG FULL-SAMPLE COMPLETED-TRADE PROFIT CONCENTRATION"
)

print(
    "=" * 76
)


display(
    full_sample_profit_concentration
)


print(
    "\nTOP 10 FULL-SAMPLE WINNING TRADES"
)

print(
    "=" * 76
)


display(
    top_10_winning_trades
)


print(
    "\nFull-sample completed-trade profit concentration analysis validated."
)

AUG FULL-SAMPLE COMPLETED-TRADE PROFIT CONCENTRATION


,Metric,Value
0,Largest Winning Trade,6.165994e+04
1,Top 1 Share of Gross Winning P&L,3.769685e-02
2,Top 3 Share of Gross Winning P&L,8.862504e-02
3,Top 5 Share of Gross Winning P&L,1.299774e-01
4,Top 10 Share of Gross Winning P&L,2.106542e-01
5,Top 1 Share of Completed Net P&L,4.357299e-02
6,Top 3 Share of Completed Net P&L,1.024398e-01
7,Top 5 Share of Completed Net P&L,1.502381e-01
8,Top 10 Share of Completed Net P&L,2.434907e-01
9,Net P&L Excluding Top 1 Winner,1.353435e+06



TOP 10 FULL-SAMPLE WINNING TRADES


,Direction,Entry_Time,Exit_Time,Holding_Bars,Net_PnL,Exit_Reason
0,Short,2026-03-23 09:05:00,2026-03-23 09:10:00,1,61659.935,Trailing Stop
1,Long,2026-01-20 13:35:00,2026-01-21 14:45:00,86,42457.635,Trailing Stop
2,Long,2026-01-29 09:05:00,2026-01-29 09:10:00,1,40844.535,Trailing Stop
3,Long,2025-10-13 09:05:00,2025-10-14 13:40:00,127,34429.435,Trailing Stop
4,Long,2025-04-21 09:05:00,2025-04-22 13:35:00,126,33209.735,Trailing Stop
5,Long,2025-12-22 09:05:00,2025-12-23 09:25:00,76,28412.435,Trailing Stop
6,Long,2025-10-09 09:05:00,2025-10-09 09:10:00,1,26208.635,Trailing Stop
7,Long,2025-04-16 09:05:00,2025-04-17 09:35:00,78,26196.635,Trailing Stop
8,Short,2026-03-19 09:05:00,2026-03-19 09:40:00,7,25815.035,Trailing Stop
9,Short,2026-02-02 09:10:00,2026-02-02 09:20:00,2,25328.535,Trailing Stop



Full-sample completed-trade profit concentration analysis validated.


## 11. Final AUG Full-Sample Summary

This section consolidates the finalized results of the AUG full-sample
in-sample benchmark.

The entire available AUG sample is optimized using the same exhaustive
parameter grid, transaction-cost assumptions, channel construction, trailing
stop logic, and Net-Profit-to-Maximum-Drawdown objective used in the
walk-forward framework.

Because the optimal parameters are selected using the entire sample, these
results are hindsight in-sample results and should not be interpreted as
independent out-of-sample evidence. Their primary role is to provide a
benchmark for comparison with the rolling walk-forward OOS results.

In [18]:
# ------------------------------------------------------------
# 1. Final optimal parameters
# ------------------------------------------------------------

final_chnlen = int(
    full_sample_optimum["ChnLen"]
)

final_stppct = float(
    full_sample_optimum["StpPct"]
)

final_objective = float(
    full_sample_optimum["Objective"]
)


# ------------------------------------------------------------
# 2. Prepare finalized full-sample path
# ------------------------------------------------------------

summary_path = (
    full_sample_results
    .copy()
)

summary_path[
    "Timestamp"
] = pd.to_datetime(
    summary_path[
        "Timestamp"
    ]
)

summary_path = (
    summary_path
    .sort_values(
        "Timestamp"
    )
    .reset_index(
        drop=True
    )
)


final_sample_start = pd.Timestamp(
    summary_path[
        "Timestamp"
    ].iloc[0]
)

final_sample_end = pd.Timestamp(
    summary_path[
        "Timestamp"
    ].iloc[-1]
)

final_observations = int(
    len(
        summary_path
    )
)


# ============================================================
# 3. PORTFOLIO PERFORMANCE
# ============================================================

final_starting_equity = float(
    INITIAL_EQUITY
)

final_ending_equity = float(
    summary_path[
        "Equity"
    ].iloc[-1]
)

final_net_profit = (
    final_ending_equity
    -
    final_starting_equity
)

final_total_return = (
    final_ending_equity
    /
    final_starting_equity
    -
    1.0
)


# ------------------------------------------------------------
# CAGR
# ------------------------------------------------------------

elapsed_years_final = (
    (
        final_sample_end
        -
        final_sample_start
    )
    .total_seconds()
    /
    (
        365.25
        *
        24.0
        *
        60.0
        *
        60.0
    )
)


final_cagr = (
    (
        final_ending_equity
        /
        final_starting_equity
    )
    **
    (
        1.0
        /
        elapsed_years_final
    )
    -
    1.0
)


# ------------------------------------------------------------
# Maximum drawdown
# ------------------------------------------------------------

equity_array = (
    summary_path[
        "Equity"
    ]
    .astype(
        float
    )
    .to_numpy()
)


running_peak_final = (
    np.maximum.accumulate(
        equity_array
    )
)


drawdown_cny_final = (
    equity_array
    -
    running_peak_final
)


drawdown_pct_final = (
    equity_array
    /
    running_peak_final
    -
    1.0
)


final_max_drawdown_cny = float(
    drawdown_cny_final.min()
)


final_max_drawdown_pct = float(
    drawdown_pct_final.min()
)


# ============================================================
# 4. DAILY SHARPE
#
# IMPORTANT:
# Use end-of-day EQUITY RETURNS, exactly as in Task K and
# Section 7. Do NOT use raw daily P&L.
# ============================================================

daily_equity_final = (
    summary_path[
        [
            "Timestamp",
            "Equity"
        ]
    ]
    .copy()
)


daily_equity_final[
    "Date"
] = (
    daily_equity_final[
        "Timestamp"
    ]
    .dt
    .normalize()
)


daily_equity_final = (
    daily_equity_final
    .groupby(
        "Date",
        as_index=True
    )[
        "Equity"
    ]
    .last()
)


# ------------------------------------------------------------
# Include initial capital as base for first daily return
# ------------------------------------------------------------

daily_equity_with_initial_final = pd.concat(
    [
        pd.Series(
            [
                INITIAL_EQUITY
            ],
            index=[
                daily_equity_final.index[0]
                -
                pd.Timedelta(
                    days=1
                )
            ],
            dtype=np.float64
        ),

        daily_equity_final.astype(
            np.float64
        )
    ]
)


daily_returns_final = (
    daily_equity_with_initial_final
    .pct_change()
    .dropna()
)


daily_returns_final = (
    daily_returns_final
    .replace(
        [
            np.inf,
            -np.inf
        ],
        np.nan
    )
    .dropna()
)


if (
    len(
        daily_returns_final
    )
    >
    1
    and
    daily_returns_final.std(
        ddof=1
    )
    >
    0
):

    final_daily_sharpe = float(
        np.sqrt(
            252.0
        )
        *
        daily_returns_final.mean()
        /
        daily_returns_final.std(
            ddof=1
        )
    )

else:

    final_daily_sharpe = np.nan


# ------------------------------------------------------------
# Calmar
# ------------------------------------------------------------

if (
    final_max_drawdown_pct
    <
    0.0
):

    final_calmar = float(
        final_cagr
        /
        abs(
            final_max_drawdown_pct
        )
    )

else:

    final_calmar = np.nan


# ============================================================
# 5. COMPLETED-TRADE STATISTICS
# ============================================================

summary_trades = (
    full_sample_trade_ledger
    .copy()
)


trade_pnl = (
    summary_trades[
        "Net_PnL"
    ]
    .astype(
        float
    )
)


winning_trade_pnl = (
    trade_pnl[
        trade_pnl
        >
        0.0
    ]
)


losing_trade_pnl = (
    trade_pnl[
        trade_pnl
        <
        0.0
    ]
)


final_completed_trades = int(
    len(
        summary_trades
    )
)


final_winning_trades = int(
    (
        trade_pnl
        >
        0.0
    ).sum()
)


final_losing_trades = int(
    (
        trade_pnl
        <
        0.0
    ).sum()
)


final_flat_trades = int(
    (
        trade_pnl
        ==
        0.0
    ).sum()
)


final_trade_win_rate = (
    final_winning_trades
    /
    final_completed_trades
)


final_gross_profit = float(
    winning_trade_pnl.sum()
)


final_gross_loss = float(
    losing_trade_pnl.sum()
)


final_profit_factor = (
    final_gross_profit
    /
    abs(
        final_gross_loss
    )
)


final_average_winner = float(
    winning_trade_pnl.mean()
)


final_average_loser = float(
    losing_trade_pnl.mean()
)


final_payoff_ratio = (
    final_average_winner
    /
    abs(
        final_average_loser
    )
)


final_expectancy = float(
    trade_pnl.mean()
)


# ============================================================
# 6. DIRECTION ATTRIBUTION
# ============================================================

final_long_pnl = float(
    summary_trades.loc[
        summary_trades[
            "Direction"
        ]
        ==
        "Long",
        "Net_PnL"
    ]
    .sum()
)


final_short_pnl = float(
    summary_trades.loc[
        summary_trades[
            "Direction"
        ]
        ==
        "Short",
        "Net_PnL"
    ]
    .sum()
)


# ============================================================
# 7. PROFIT CONCENTRATION
# ============================================================

sorted_winners_final = (
    winning_trade_pnl
    .sort_values(
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


def winner_share_of_completed_net(
    n
):

    return float(
        sorted_winners_final
        .head(
            n
        )
        .sum()
        /
        final_net_profit
    )


final_top1_share = (
    winner_share_of_completed_net(
        1
    )
)


final_top3_share = (
    winner_share_of_completed_net(
        3
    )
)


final_top5_share = (
    winner_share_of_completed_net(
        5
    )
)


final_top10_share = (
    winner_share_of_completed_net(
        10
    )
)


final_net_ex_top10 = float(
    final_net_profit
    -
    sorted_winners_final
    .head(
        10
    )
    .sum()
)


# ============================================================
# 8. PARAMETER-BOUNDARY DIAGNOSTICS
# ============================================================

chnlen_at_lower_bound = bool(
    final_chnlen
    ==
    int(
        CHNLEN_RANGE[0]
    )
)


chnlen_at_upper_bound = bool(
    final_chnlen
    ==
    int(
        CHNLEN_RANGE[-1]
    )
)


stppct_at_lower_bound = bool(
    np.isclose(
        final_stppct,
        float(
            STPPCT_RANGE[0]
        )
    )
)


stppct_at_upper_bound = bool(
    np.isclose(
        final_stppct,
        float(
            STPPCT_RANGE[-1]
        )
    )
)


# ============================================================
# 9. FINAL SUMMARY TABLE
# ============================================================

task_l_final_summary = pd.DataFrame(
    {
        "Metric": [
            "Market",
            "Analysis Type",
            "Sample Start",
            "Sample End",
            "Observations",

            "ChnLen",
            "StpPct",
            "Objective",

            "ChnLen at Lower Bound",
            "ChnLen at Upper Bound",
            "StpPct at Lower Bound",
            "StpPct at Upper Bound",

            "Starting Equity",
            "Ending Equity",
            "Net Profit",
            "Total Return",
            "CAGR",

            "Maximum Drawdown (CNY)",
            "Maximum Drawdown (%)",

            "Daily Sharpe",
            "Calmar",

            "Completed Trades",
            "Trade Win Rate",
            "Profit Factor",
            "Payoff Ratio",
            "Expectancy per Trade",

            "Long Net P&L",
            "Short Net P&L",

            "Top 1 Trade Share of Net P&L",
            "Top 3 Trades Share of Net P&L",
            "Top 5 Trades Share of Net P&L",
            "Top 10 Trades Share of Net P&L",

            "Net P&L Excluding Top 10 Winners"
        ],

        "Value": [
            "AUG / SHFE Gold Futures",
            "Full-Sample In-Sample Optimization",
            final_sample_start,
            final_sample_end,
            final_observations,

            final_chnlen,
            final_stppct,
            final_objective,

            chnlen_at_lower_bound,
            chnlen_at_upper_bound,
            stppct_at_lower_bound,
            stppct_at_upper_bound,

            final_starting_equity,
            final_ending_equity,
            final_net_profit,
            final_total_return,
            final_cagr,

            final_max_drawdown_cny,
            final_max_drawdown_pct,

            final_daily_sharpe,
            final_calmar,

            final_completed_trades,
            final_trade_win_rate,
            final_profit_factor,
            final_payoff_ratio,
            final_expectancy,

            final_long_pnl,
            final_short_pnl,

            final_top1_share,
            final_top3_share,
            final_top5_share,
            final_top10_share,

            final_net_ex_top10
        ]
    }
)


# ============================================================
# 10. FINAL CONSISTENCY VALIDATION
# ============================================================

portfolio_accounting_error = abs(
    final_net_profit
    -
    float(
        summary_path[
            "PnL"
        ].sum()
    )
)


trade_accounting_error = abs(
    float(
        trade_pnl.sum()
    )
    -
    final_net_profit
)


direction_accounting_error = abs(
    (
        final_long_pnl
        +
        final_short_pnl
    )
    -
    final_net_profit
)


# ------------------------------------------------------------
# Reconcile final performance with Section 7
# ------------------------------------------------------------

section7_sharpe_error = abs(
    final_daily_sharpe
    -
    float(
        daily_sharpe
    )
)


section7_cagr_error = abs(
    final_cagr
    -
    float(
        cagr
    )
)


section7_mdd_pct_error = abs(
    final_max_drawdown_pct
    -
    float(
        maximum_drawdown_pct
    )
)


section7_calmar_error = abs(
    final_calmar
    -
    float(
        calmar
    )
)


# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

assert (
    portfolio_accounting_error
    <
    1e-6
)


assert (
    trade_accounting_error
    <
    1e-6
)


assert (
    direction_accounting_error
    <
    1e-6
)


assert (
    section7_sharpe_error
    <
    1e-12
)


assert (
    section7_cagr_error
    <
    1e-12
)


assert (
    section7_mdd_pct_error
    <
    1e-12
)


assert (
    section7_calmar_error
    <
    1e-12
)


assert (
    final_completed_trades
    ==
    len(
        full_sample_trade_ledger
    )
)


assert (
    final_winning_trades
    +
    final_losing_trades
    +
    final_flat_trades
    ==
    final_completed_trades
)


assert (
    final_chnlen
    in
    set(
        CHNLEN_RANGE
    )
)


assert np.any(
    np.isclose(
        STPPCT_RANGE,
        final_stppct
    )
)


assert (
    final_long_pnl
    >
    0.0
)


assert (
    final_short_pnl
    >
    0.0
)


# ============================================================
# 11. DISPLAY
# ============================================================

print(
    "TASK L — FINAL AUG FULL-SAMPLE SUMMARY"
)

print(
    "=" * 76
)


display(
    task_l_final_summary
)


print(
    "\nFINAL TASK L VALIDATION"
)

print(
    "=" * 76
)


print(
    f"Portfolio accounting error    : "
    f"{portfolio_accounting_error:.12f}"
)


print(
    f"Trade-ledger accounting error : "
    f"{trade_accounting_error:.12f}"
)


print(
    f"Direction attribution error   : "
    f"{direction_accounting_error:.12f}"
)


print(
    f"Section 7 Sharpe error        : "
    f"{section7_sharpe_error:.12f}"
)


print(
    f"Section 7 CAGR error          : "
    f"{section7_cagr_error:.12f}"
)


print(
    f"Section 7 MDD% error          : "
    f"{section7_mdd_pct_error:.12f}"
)


print(
    f"Section 7 Calmar error        : "
    f"{section7_calmar_error:.12f}"
)


print(
    f"ChnLen boundary               : "
    f"lower={chnlen_at_lower_bound}, "
    f"upper={chnlen_at_upper_bound}"
)


print(
    f"StpPct boundary               : "
    f"lower={stppct_at_lower_bound}, "
    f"upper={stppct_at_upper_bound}"
)


print(
    "\nAll final Task L consistency checks passed."
)

TASK L — FINAL AUG FULL-SAMPLE SUMMARY


,Metric,Value
0,Market,AUG / SHFE Gold Futures
1,Analysis Type,Full-Sample In-Sample Optimization
2,Sample Start,2018-05-03 09:05:00
3,Sample End,2026-04-10 15:00:00
4,Observations,138744
5,ChnLen,710
6,StpPct,0.005
7,Objective,85.657026
8,ChnLen at Lower Bound,False
9,ChnLen at Upper Bound,False



FINAL TASK L VALIDATION
Portfolio accounting error    : 0.000000000000
Trade-ledger accounting error : 0.000000000000
Direction attribution error   : 0.000000000000
Section 7 Sharpe error        : 0.000000000000
Section 7 CAGR error          : 0.000000000000
Section 7 MDD% error          : 0.000000000000
Section 7 Calmar error        : 0.000000000000
ChnLen boundary               : lower=False, upper=False
StpPct boundary               : lower=True, upper=False

All final Task L consistency checks passed.
